# Finwise Scribe — GGUF Conversion & Model Export (Kaggle) — v2

**Purpose:** Converts the LoRA adapter (trained in `01_FinwiseScribe_Training.ipynb` on Colab) into a quantized GGUF file suitable for on-device inference with Ollama.

**SLM v2 changes from v1:**
- Model: **Qwen2-1.5B** (was Llama-3-8B). Adapter ZIP is `finwise_scribe_adapter_v2.zip`.
- Export target: `finwise_scribe_v2.gguf`

**Workflow:**
1. Install dependencies
2. Download the LoRA adapter v2 ZIP from Google Drive
3. Load Qwen2-1.5B + adapter and quantize to GGUF (Q4_K_M)
4. Move the final GGUF to Kaggle's output directory for download
5. (Optional) Upload the GGUF to Hugging Face Hub as `finwise-scribe-model-v2`
6. Generate the Ollama Modelfile

**Target Runtime:** Kaggle — GPU T4 x2 recommended  
**Disk Strategy:** All intermediate files (up to 8 GB for 1.5B model) are written to `/tmp` to avoid Kaggle's 20 GB `/kaggle/working` disk limit.

In [ ]:
# ==============================================================
# CELL 1: Environment Setup
#
# Installs Unsloth and supporting packages required for loading the
# Qwen2-1.5B LoRA adapter and performing GGUF quantization.
# NOTE: Designed for Kaggle — run once per session.
# ==============================================================

import os
import shutil

print("Installing dependencies...")
# Unsloth supports Qwen2 from version ≥ 2024.6 — no extra args needed
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes --quiet
!pip install gdown huggingface_hub --quiet

print("Setup complete.")

Kurulum yapılıyor...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 28.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/199.3 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 36.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.1 MB/s eta 0:00:00:00:0100:0

## Step 2 — Download, Convert & Export
Downloads the LoRA adapter from Google Drive, merges it with the base model, quantizes to GGUF (Q4_K_M), and moves the result to `/kaggle/working` for download.

In [ ]:
# ==============================================================
# CELL 2: Download Adapter → Load Qwen2-1.5B → Quantize → Export GGUF
#
# SLM v2: uses Qwen2-1.5B base model + v2 LoRA adapter.
# Intermediate files are ~8 GB (vs ~16 GB for Llama-3-8B), leaving
# more headroom on Kaggle's T4 for a faster conversion.
#
# Disk strategy:
#   - All intermediate files are written to /tmp.
#   - Only the final GGUF (~900 MB for Q4_K_M 1.5B) is copied to /kaggle/working.
#
# TODO: Replace DRIVE_FILE_ID with the Google Drive file ID of the
#       v2 adapter ZIP produced by 01_FinwiseScribe_Training.ipynb on Colab.
#       (File name: finwise_scribe_adapter_v2.zip)
# ==============================================================

from unsloth import FastLanguageModel
import os
import shutil
import glob

# --- Step 1: Route all heavy I/O through /tmp ---
os.chdir("/tmp")
print("Working directory set to /tmp (bypasses Kaggle disk quota).")

# --- Step 2: Download LoRA adapter v2 ZIP from Google Drive ---
# TODO: Replace with your own Drive file ID for finwise_scribe_adapter_v2.zip
DRIVE_FILE_ID = "YOUR_V2_ADAPTER_DRIVE_FILE_ID"
OUTPUT_ZIP    = "finwise_scribe_adapter_v2.zip"
DRIVE_URL     = f"https://drive.google.com/uc?id={DRIVE_FILE_ID}"

print(f"Downloading adapter archive v2 (Drive ID: {DRIVE_FILE_ID})...")
!gdown {DRIVE_URL} -O {OUTPUT_ZIP}

# --- Step 3: Extract adapter archive ---
print("Extracting adapter archive...")
!unzip -o {OUTPUT_ZIP} -d finwise_adapter_v2

# Locate the adapter root (Unsloth may nest one level inside the ZIP)
adapter_dir = "/tmp/finwise_adapter_v2"
for root, dirs, files in os.walk("/tmp/finwise_adapter_v2"):
    if "adapter_config.json" in files:
        adapter_dir = root
        break
print(f"Adapter root located at: {adapter_dir}")

# --- Step 4: Load Qwen2-1.5B + LoRA adapter (4-bit) ---
# Qwen2-1.5B is ~1.5B parameters — loads in <2 min on Kaggle T4.
print("Loading Qwen2-1.5B with v2 LoRA adapter (4-bit)...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = adapter_dir,
    max_seq_length = 512,   # Match the training max_seq_length from 01_FinwiseScribe_Training.ipynb
    dtype          = None,
    load_in_4bit   = True,
)

# --- Step 5: Quantize and export to GGUF (Q4_K_M) ---
# Q4_K_M for 1.5B model produces a ~900 MB GGUF — well within
# Kaggle's output limit and suitable for 4 GB VRAM edge deployment.
GGUF_OUTPUT_DIR = "temp_model_gguf_v2"
print("Converting to GGUF (q4_k_m) — this typically takes 3–5 min for 1.5B...")
model.save_pretrained_gguf(GGUF_OUTPUT_DIR, tokenizer, quantization_method="q4_k_m")

# --- Step 6: Locate the GGUF output file ---
# Unsloth appends the quantization suffix dynamically; glob for safety.
gguf_candidates = glob.glob(f"/tmp/{GGUF_OUTPUT_DIR}/*.gguf")
if not gguf_candidates:
    gguf_candidates = glob.glob("/tmp/**/*.gguf", recursive=True)

if not gguf_candidates:
    print("[ERROR] GGUF file was not produced. Check the conversion logs above.")
    print("Run Cell 3 (GGUF File Recovery) to scan /tmp for the output file.")
else:
    source_file = gguf_candidates[0]
    dest_file   = "/kaggle/working/finwise_scribe_v2.gguf"
    print("Moving GGUF to Kaggle output directory...")
    shutil.copy(source_file, dest_file)
    size_mb = os.path.getsize(dest_file) / (1024 ** 2)
    print(f"\n[OK] GGUF conversion successful.")
    print(f"File : {dest_file}  ({size_mb:.0f} MB)")
    print("Download it from the 'Output' panel on the right side of Kaggle.")
    print("Next step: Run Cell 4 (Optional) to upload to HF Hub as finwise-scribe-model-v2.")

Çalışma dizini /tmp olarak değiştirildi.
Dosya indiriliyor (ID: 1qDta8PKGAWzanCn8_HXcZY2YRwwxWVJp)...
Downloading...
From (original): https://drive.google.com/uc?id=1qDta8PKGAWzanCn8_HXcZY2YRwwxWVJp
From (redirected): https://drive.google.com/uc?id=1qDta8PKGAWzanCn8_HXcZY2YRwwxWVJp&confirm=t&uuid=1c86150c-ebe8-4a41-882a-6fb630bc6b3f
To: /tmp/finwise_scribe_adapter_v1.zip
100%|████████████████████████████████████████| 158M/158M [00:01<00:00, 86.1MB/s]
Zip açılıyor...
Archive:  finwise_scribe_adapter_v1.zip
   creating: finwise_adapter/finwise_scribe_adapter/
  inflating: finwise_adapter/finwise_scribe_adapter/tokenizer_config.json  
  inflating: finwise_adapter/finwise_scribe_adapter/adapter_config.json  
  inflating: finwise_adapter/finwise_scribe_adapter/README.md  
  inflating: finwise_adapter/finwise_scribe_adapter/adapter_model.safetensors  
  inflating: finwise_adapter/finwise_scribe_adapter/special_tokens_map.json  
  inflating: finwise_adapter/finwise_scribe_adapter/tokenizer.js

/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


==((====))==  Unsloth 2025.11.3: Fast Llama patching. Transformers: 4.57.1.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.11.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


💾 GGUF formatına dönüştürülüyor (q4_k_m)...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [01:00<00:00, 15.03s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:10<00:00, 32.60s/it]


Unsloth: Merge process complete. Saved to `/tmp/temp_model_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: All required system packages already installed!
Unsloth: Install llama.cpp and building - please wait 1 to 3 minutes
Unsloth: Cloning llama.cpp repository
Unsloth: Install GGUF and other packages
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['llama-3-8b.F16.gguf']
Unsloth: [2] Conv

## Step 3 — GGUF File Recovery (Fallback)
Run this cell only if Cell 2 completed but the expected output file was not found at the default path. Scans all of `/tmp` and moves the first `.gguf` file found.

In [ ]:
# ==============================================================
# CELL 3: GGUF File Recovery (Fallback)
#
# If Cell 2 completed successfully but the expected output path
# was not found (e.g., Unsloth used a different filename), this
# cell scans all of /tmp and moves the first discovered .gguf
# file to the Kaggle output directory.
# ==============================================================

import os
import shutil

print("Scanning /tmp for GGUF output files...")

found_file = None
for root, dirs, files in os.walk("/tmp"):
    for file in files:
        if file.endswith(".gguf"):
            found_file = os.path.join(root, file)
            print(f"[FOUND] {found_file}")
            break
    if found_file:
        break

if found_file:
    dest_path = "/kaggle/working/finwise_scribe_v1.gguf"
    print(f"Moving to Kaggle output: {dest_path}")
    shutil.copy(found_file, dest_path)
    print("\n[OK] File is ready. Refresh the 'Output' panel in Kaggle to download.")
else:
    print("[ERROR] No GGUF file found anywhere in /tmp.")
    print("Possible causes:")
    print("  - The quantization step in Cell 2 failed silently.")
    print("  - The model was saved to a different directory.")
    print("\nDEBUG — /tmp top-level contents:")
    print(os.listdir("/tmp"))

🔍 Kayıp GGUF dosyası aranıyor...
✅ BULUNDU: /tmp/llama-3-8b.Q4_K_M.gguf
📦 Dosya taşınıyor: /kaggle/working/finwise_scribe_v1.gguf ...

🎉 İŞLEM TAMAM! Sağ taraftaki 'Output' panelini yenile (Refresh).
Dosyayı oradan indirebilirsin.


## Step 4 — Upload to Hugging Face Hub (Optional)
Uploads the quantized GGUF to a Hugging Face model repository. Requires a WRITE-scoped HF token. Set `HF_TOKEN` before running.

In [ ]:
# ==============================================================
# CELL 4: Upload GGUF v2 to Hugging Face Hub (Optional)
#
# Uploads the Qwen2-1.5B GGUF to a dedicated v2 repository on HF.
# Target repo: <your-username>/finwise-scribe-model-v2
#
# Prerequisites:
#   - A Hugging Face account with a WRITE-scoped access token.
#   - Token creation: https://huggingface.co/settings/tokens
#
# TODO: Set HF_TOKEN before running this cell.
# ==============================================================

from huggingface_hub import HfApi
import os

# TODO: Paste your Hugging Face WRITE token here
HF_TOKEN = ""  # Required — leave empty to skip upload

try:
    api       = HfApi(token=HF_TOKEN)
    user_info = api.whoami()
    username  = user_info["name"]
    print(f"Authenticated as: {username}")

    # v2 has a dedicated repo to preserve the v1 model
    repo_name = "finwise-scribe-model-v2"
    repo_id   = f"{username}/{repo_name}"
    print(f"Preparing repository: {repo_id}")
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

    # Resolve the v2 GGUF path
    file_path = "/kaggle/working/finwise_scribe_v2.gguf"
    if not os.path.exists(file_path):
        print(f"[WARNING] File not found at: {file_path}")
        tmp_candidates = [f for f in __import__("glob").glob("/tmp/**/*.gguf", recursive=True)]
        if tmp_candidates:
            file_path = tmp_candidates[0]
            print(f"Using fallback path: {file_path}")
        else:
            raise FileNotFoundError(
                "GGUF file not found. Ensure Cell 2 or Cell 3 completed successfully."
            )

    size_mb = os.path.getsize(file_path) / (1024 ** 2)
    print(f"Uploading: {file_path} ({size_mb:.0f} MB)  →  huggingface.co/{repo_id}")
    print("Upload may take 1–3 minutes for a ~900 MB file...")
    api.upload_file(
        path_or_fileobj = file_path,
        path_in_repo    = "finwise_scribe_v2.gguf",
        repo_id         = repo_id,
        repo_type       = "model",
    )

    print("\n[OK] Upload successful.")
    print(f"Model URL: https://huggingface.co/{repo_id}/blob/main/finwise_scribe_v2.gguf")

except Exception as e:
    print(f"\n[ERROR] Upload failed: {e}")
    print("Tip: Ensure your HF token has WRITE permissions.")
    print("     Generate one at: https://huggingface.co/settings/tokens")

👤 Giriş yapıldı: MV17
🔨 Repo hazırlanıyor: MV17/finwise-scribe-model ...
🚀 Yükleme başlıyor: /kaggle/working/finwise_scribe_v1.gguf -> Hugging Face Hub
Bu işlem dosya boyutuna (4-5GB) göre 2-5 dakika sürebilir...

✅ BAŞARILI! Dosyan bulutta güvende.
📥 İndirme Linkin: https://huggingface.co/MV17/finwise-scribe-model/blob/main/finwise_scribe_v1.gguf


## Step 5 — Generate Ollama Modelfile
Writes the `Modelfile` required by `ollama create` to serve the GGUF model. The Modelfile encodes the system prompt and inference parameters that match the production settings in `finwise_scribe/models/Modelfile`.

In [ ]:
# ==============================================================
# CELL 5: Generate Ollama Modelfile (v2)
#
# Writes the Modelfile needed to register finwise_scribe_v2.gguf
# with Ollama. After downloading from Kaggle, run:
#
#   docker exec -it finwise_ollama ollama create finwise_scribe_v2 \
#       -f /models/Modelfile
#
# Parameters match production config in llm_service/app/core/config.py.
# Temperature=0 + top_k=1 enforces deterministic token prediction.
# ==============================================================

MODELFILE_CONTENT = """\
FROM ./finwise_scribe_v2.gguf

# Inference parameters — must match llm_service/app/core/config.py
PARAMETER temperature 0.0
PARAMETER top_k 1
PARAMETER top_p 1.0
PARAMETER repeat_penalty 1.1
PARAMETER num_ctx 512

SYSTEM \"\"\"
You are FinwiseScribe v2, a neuro-symbolic market forecasting engine.
Your only job is to predict the next token in a sequence of symbolic
market state tokens.

Token vocabulary (fixed semantic bins):
  Price : P_CRASH | P_LOW | P_STABLE | P_HIGH | P_SURGE
  Volume: V_DROUGHT | V_LOW | V_STABLE | V_HIGH | V_SURGE
  Composite example: P_SURGE_V_HIGH

Rules:
- Respond with exactly ONE composite token. No words, no explanation.
- Do NOT hallucinate numeric prices or percentages.
- Let the token sequence speak for itself.
\"\"\"
"""

MODELFILE_PATH = "/kaggle/working/Modelfile"

with open(MODELFILE_PATH, "w") as f:
    f.write(MODELFILE_CONTENT)

print(f"[OK] Modelfile (v2) written to: {MODELFILE_PATH}")
print()
print("--- Modelfile contents ---")
print(MODELFILE_CONTENT)
print("--- Next Steps ---")
print("1. Download finwise_scribe_v2.gguf and Modelfile from the Kaggle Output panel.")
print("2. Place both files in finwise_scribe/models/")
print("3. Run:")
print("     docker exec -it finwise_ollama ollama create finwise_scribe_v2 -f /models/Modelfile")